<a href="https://colab.research.google.com/github/boss-defender/Smart-Fine-Tune/blob/main/SmartFineTuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# ⚡ UNSLOTH AUTO-TRAINER (FAIL-PROOF SINGLE CELL)
# ==========================================

import os
import hashlib
import json
import torch

# --- STEP 1: VERIFY GPU ---
print("🔍 Checking Hardware...")
if not torch.cuda.is_available():
    raise SystemError("❌ No GPU found! Go to Runtime > Change runtime type > T4 GPU")
print(f"✅ GPU DETECTED: {torch.cuda.get_device_name(0)}! Hardware verified.\n")

# --- STEP 2: MOUNT GOOGLE DRIVE ---
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive connected!\n")

# --- STEP 3: INSTALL ALL REQUIRED DEPENDENCIES ---
print("🔄 Installing Unsloth, Unsloth Zoo & Core Libraries...")
!pip install --quiet --upgrade pip setuptools wheel
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth_zoo
!pip install --quiet "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets huggingface_hub hf_transfer
print("✅ Packages installed successfully!\n")

# --- STEP 4: CONFIGURATION & EXPERIMENT ISOLATION ---
MODEL_NAME = "Qwen/Qwen3-1.7B"                    #@param {type:"string"}
DATASET_NAME = "bespokelabs/Bespoke-Stratos-35k"    #@param {type:"string"}
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# Hyperparameters
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
SAVE_STEPS = 25

# Generate SHA-256 Hash for Experiment Isolation
config_str = f"{MODEL_NAME}_{DATASET_NAME}_{LEARNING_RATE}_{BATCH_SIZE}_{GRAD_ACCUM}"
config_hash = hashlib.sha256(config_str.encode()).hexdigest()[:8]
folder_name = f"{MODEL_NAME.split('/')[-1]}__{DATASET_NAME.split('/')[-1]}__{config_hash}".replace(".", "_")

DRIVE_BASE_DIR = "/content/drive/MyDrive/unsloth_checkpoints"
RUN_DIR = os.path.join(DRIVE_BASE_DIR, folder_name)
os.makedirs(RUN_DIR, exist_ok=True)

print(f"📂 UNIQUE RUN DIRECTORY: {folder_name}")
print(f"🔒 SHA-256 Config Hash: {config_hash}\n")

# Save run_config.json metadata lock
config_file = os.path.join(RUN_DIR, "run_config.json")
run_metadata = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "hash": config_hash
}
with open(config_file, "w") as f:
    json.dump(run_metadata, f, indent=2)

# --- STEP 5: LOAD MODEL & TOKENIZER ---
print(f"📥 Loading Base Model: {MODEL_NAME}...")
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)

# Apply Fast LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- STEP 6: LOAD & FORMAT DATASET ---
print(f"📊 Loading Dataset: {DATASET_NAME}...")
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, split="train")

def format_prompts(batch):
    formatted_texts = []

    # Handle 'conversations' (Bespoke-Stratos style)
    if "conversations" in batch:
        for conv in batch["conversations"]:
            text = ""
            for msg in conv:
                role = msg.get("from", msg.get("role", "user"))
                content = msg.get("value", msg.get("content", ""))
                if role in ["human", "user"]:
                    text += f"<|im_start|>user\n{content}<|im_end|>\n"
                elif role in ["gpt", "assistant"]:
                    text += f"<|im_start|>assistant\n{content}<|im_end|>\n"
                elif role == "system":
                    text += f"<|im_start|>system\n{content}<|im_end|>\n"
            formatted_texts.append(text)

    # Handle 'messages' (OpenAI style)
    elif "messages" in batch:
        for msg_list in batch["messages"]:
            text = ""
            for msg in msg_list:
                role = msg.get("role", "user")
                content = msg.get("content", "")
                text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
            formatted_texts.append(text)

    # Handle 'instruction' / 'output' (Alpaca style)
    elif "instruction" in batch:
        for inst, inp, outp in zip(batch["instruction"], batch.get("input", [""] * len(batch["instruction"])), batch["output"]):
            input_part = f"\n{inp}" if inp else ""
            text = f"<|im_start|>user\n{inst}{input_part}<|im_end|>\n<|im_start|>assistant\n{outp}<|im_end|>\n"
            formatted_texts.append(text)

    return {"text": formatted_texts}

dataset = dataset.map(format_prompts, batched=True)
print("✅ Dataset formatted successfully with 'text' column!\n")

# --- STEP 7: SETUP TRAINER & CHECKPOINT RECOVERY ---
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Check Google Drive for existing checkpoints to auto-resume
checkpoints = [d for d in os.listdir(RUN_DIR) if d.startswith("checkpoint-")]
resume_from_checkpoint = None

if checkpoints:
    latest_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
    resume_from_checkpoint = os.path.join(RUN_DIR, latest_checkpoint)
    print(f"🔄 AUTO-RESUMING from checkpoint: {latest_checkpoint}")
else:
    print("🚀 STARTING FRESH TRAINING RUN...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = RUN_DIR,
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        save_total_limit = 3,
    ),
)

# --- STEP 8: LAUNCH TRAINING ---
print("🔥 Training in progress...")
trainer.train(resume_from_checkpoint=resume_from_checkpoint)
print("🎉 Training Completed!\n")

# --- STEP 9: EXPORT 16-BIT MERGED BASE MODEL ---
EXPORT_DIR = os.path.join(RUN_DIR, "merged_16bit_model")
print(f"📦 Exporting 16-bit merged base model to Google Drive: {EXPORT_DIR}...")
model.save_pretrained_merged(EXPORT_DIR, tokenizer, save_method = "merged_16bit")
print("✅ Model exported! All files are safely sitting in your Google Drive. 🚀")

In [ ]:
# ==============================================================================
# LIGHTWEIGHT HF UPLOADER (NO UNSLOTH NEEDED! 🚀)
# ==============================================================================
!pip install --quiet huggingface_hub

from huggingface_hub import HfApi
import os

# 1. CONFIGURATION PANEL 🎛️
HF_WRITE_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}
HF_REPO_NAME = "your-username/my-finetuned-model"        #@param {type:"string"}
FOLDER_PATH = "/content/merged_16bit_model"             #@param {type:"string"}
REPO_VISIBILITY = "public"                             #@param ["private", "public"]

# 2. VERIFY FOLDER EXISTS
if not os.path.exists(FOLDER_PATH):
    raise FileNotFoundError(f"❌ Cannot find folder at: {FOLDER_PATH}")

# 3. INITIALIZE HF API & CREATE REPO
api = HfApi(token=HF_WRITE_TOKEN)

print(f"📁 Preparing Hugging Face repository '{HF_REPO_NAME}' ({REPO_VISIBILITY.upper()})...")
api.create_repo(
    repo_id=HF_REPO_NAME,
    private=(REPO_VISIBILITY == "private"),
    exist_ok=True,
    repo_type="model"
)

# 4. PUSH FOLDER TO HUGGING FACE
print(f"🚀 Uploading all files from '{FOLDER_PATH}'...")
api.upload_folder(
    folder_path=FOLDER_PATH,
    repo_id=HF_REPO_NAME,
    repo_type="model",
)

print(f"\n🎉 BOOM! Your model is live at: https://huggingface.co/{HF_REPO_NAME}")

In [ ]:
# @title ⬇️ 3. Auto-Download Model
# @markdown Pushes your zip file straight into your browser download queue! 🚀

import os
from google.colab import files

zip_file = "/content/merged_16bit_model" #@param {type:"string"}

if os.path.exists(zip_file):
  print("⬇️ Triggering browser download now...")
  files.download(zip_file)
else:
  print(
      "❌ Zip file missing! Make sure to hit play on Cell 3 (Create Zip"
      " Archive) first."
  )